# a quick test for our larger k562 model with pooling

Training on several cell types, want to see good model size

In [1]:
import sys
sys.path.append('/data1/lesliec/sarthak/caduceus/')
from src.models.nn.sampling import DownsampleStack, UpsampleStack
from src.models.sequence.striped_backbone import StripedMambaBackbone
import torch

model = StripedMambaBackbone(
    d_model=512,
    n_blocks=4,
    d_in=128,
    global_pooling=256,
    transformer_pooling=4,
    mode='striped',
    ssm_per_transformer=3,
    # Hydra config
    d_state=64,
    d_conv=7,
    expand=2,
    headdim=64,
    ngroups=8,
    chunk_size=256,
    use_mem_eff_path=True,
    # Transformer config
    head_dim=64,
    use_rope=True,
    use_flash_attn=True,
    use_gating=True,
    attention_dropout=0.0,
    use_enformer_bias=False,
    pos_emb_dim=32,
    # Shared config
    expansion_factor=2,
    norm='rms',
    dropout=0.1,
    mlp_activation='gelu',
    mlp_dropout=0.0,
    fused_mlp=False,
    hydra_use_mlp=False,
    checkpoint_blocks=False,
    residual_in_fp32=True,
    rescale_prenorm_residual=True,
    zero_linear_biases=True,
    # Sampling config
    downsample_kernel_size=5,
    global_sampling_channel_scale=2,
    global_sampling_start_channels=None,
    global_sampling_grow_channels=None,
    transformer_sampling_channel_scale=1,
    transformer_sampling_start_channels=None,
    transformer_sampling_grow_channels=None,
    upsample_residual_scale_init=0.9,
    sampling_norm_type='rms',
    sampling_use_weight_std=True,
    sampling_checkpoint=True,)
model.to('cuda').bfloat16()

StripedMambaBackbone(
  (conv_block): ConvBlock(
    (norm): RMSBatchNorm1d()
    (act): GELU(approximate='none')
    (op): StandardizedConv1d(128, 128, kernel_size=(5,), stride=(1,), padding=(2,))
  )
  (global_down): DownsampleStack(
    (layers): ModuleList(
      (0): DownsampleLayer(
        (downres_increase): ConvBlock(
          (norm): RMSBatchNorm1d()
          (act): GELU(approximate='none')
          (op): StandardizedConv1d(128, 182, kernel_size=(5,), stride=(1,), padding=(2,))
        )
        (downres_refine): ConvBlock(
          (norm): RMSBatchNorm1d()
          (act): GELU(approximate='none')
          (op): StandardizedConv1d(182, 182, kernel_size=(5,), stride=(1,), padding=(2,))
        )
        (pool): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      )
      (1): DownsampleLayer(
        (downres_increase): ConvBlock(
          (norm): RMSBatchNorm1d()
          (act): GELU(approximate='none')
          (op): StandardizedConv1d(18

In [2]:
#print num param and make it human readable
num_params = sum(p.numel() for p in model.parameters())
print(f'Number of parameters: {num_params:,}')

Number of parameters: 94,670,574


In [3]:
model.conv_block

ConvBlock(
  (norm): RMSBatchNorm1d()
  (act): GELU(approximate='none')
  (op): StandardizedConv1d(32, 32, kernel_size=(5,), stride=(1,), padding=(2,))
)

In [ ]:
#let's do the forward one step at a time and see memory usage and output shapes
#wait this is batch size 2 as well!!
x = torch.randn(2, 2**21, 128, device='cuda', dtype=torch.bfloat16)
print(f'Input shape: {x.shape}, memory usage: {torch.cuda.memory_allocated() / 1e9:.2f} GB')
x = model.conv_block(x) + x
global_intermediates = None
if model.global_down is not None:
    x, global_intermediates = model.global_down(x)
print(f'After global downsample: {x.shape}, memory usage: {torch.cuda.memory_allocated() / 1e9:.2f} GB')
for i in range(model.n_blocks):
    if model.mode == 'striped':
        # Hydra at global-pooled resolution
        for j in range(model.ssm_per_transformer):
            x = model.hydra_blocks[i * model.ssm_per_transformer + j](x)
        # Downsample for transformer
        if model.transformer_pooling > 1:
            x, trans_intermediates = model.trans_down[i](x)
        # Transformer at further compressed resolution
        x = model.transformer_blocks[i](x)
        # Upsample back with skip connections
        if model.transformer_pooling > 1:
            x = model.trans_up[i](x, trans_intermediates)
    print(f'After block {i}: {x.shape}, memory usage: {torch.cuda.memory_allocated() / 1e9:.2f} GB')
x = model.global_up(x, global_intermediates)
print(f'After global up: {x.shape}, memory usage: {torch.cuda.memory_allocated() / 1e9:.2f} GB')

#for this example we're looking at 18 GB when checkpoint vs like 53 jesus, definitely checkpoint lmfao!

Input shape: torch.Size([2, 2097152, 128]), memory usage: 1.43 GB
After global downsample: torch.Size([2, 8192, 512]), memory usage: 14.13 GB
After block 0: torch.Size([2, 8192, 512]), memory usage: 15.04 GB
After block 1: torch.Size([2, 8192, 512]), memory usage: 15.94 GB
After block 2: torch.Size([2, 8192, 512]), memory usage: 16.84 GB
After block 3: torch.Size([2, 8192, 512]), memory usage: 17.74 GB
After global up: torch.Size([2, 2097152, 128]), memory usage: 20.76 GB


In [ ]:
import time
x = torch.randn(2, 2**21, 128, device='cuda', dtype=torch.bfloat16)
# Warmup (compile happens on first call)   
for _ in range(2):
    out = model(x)
                                                                                                                     
torch.cuda.synchronize()
t0 = time.perf_counter()
for _ in range(10):                                                                                                                                             
    out = model(x)
torch.cuda.synchronize()                                                                                                                                        
print((time.perf_counter() - t0) / 10 * 1000, "ms")
#relatively fast!

795.3941458370537 ms


In [4]:
out.shape

torch.Size([2, 2097152, 128])

In [5]:
#forward pass takes 60 GB lmao. Let's see if it can do forward and backward
import gc
torch.cuda.empty_cache()
gc.collect()

0

In [ ]:
#time it forward and backward
x = torch.randn(2, 2**21, 128, device='cuda', dtype=torch.bfloat16)
t0 = time.perf_counter()
for _ in range(10):
    out = model(x)
    out.mean().backward()
torch.cuda.synchronize()
print((time.perf_counter() - t0) / 10 * 1000, "ms")
#52GB now it seems?
#why is backward so slow? let's double check this

9212.873939005658 ms


In [ ]:
#less than 60 before
import gc
torch.cuda.empty_cache()
gc.collect()

x = torch.randn(2, 2**21, 128, device='cuda', dtype=torch.bfloat16)
t0 = time.perf_counter()
for _ in range(10):
    out = model(x)
    out.mean().backward()
torch.cuda.synchronize()
print((time.perf_counter() - t0) / 10 * 1000, "ms")
#yeah notably faster, I think 1 second forward, 2 seconds backward makes sense!

3365.5625908169895 ms


In [2]:
#can we do batchsize 4
import gc
import time
torch.cuda.empty_cache()
gc.collect()

x = torch.randn(3, 2**21, 128, device='cuda', dtype=torch.bfloat16)
#warmup
for _ in range(2):
    out = model(x)

t0 = time.perf_counter()
for _ in range(10):
    out = model(x)
    out.mean().backward()
torch.cuda.synchronize()
print((time.perf_counter() - t0) / 10 * 1000, "ms")

6876.083004800603 ms


In [3]:
#it does technically run, but training I think use batch size 2!
import gc
import time
torch.cuda.empty_cache()
gc.collect()

x = torch.randn(2, 2**21, 128, device='cuda', dtype=torch.bfloat16)
#warmup
for _ in range(2):
    out = model(x)

t0 = time.perf_counter()
for _ in range(10):
    out = model(x)
    out.mean().backward()
torch.cuda.synchronize()
print((time.perf_counter() - t0) / 10 * 1000, "ms")

3415.485413884744 ms


In [ ]:
#final test let's see if it works with 1 less pooling
import sys
sys.path.append('/data1/lesliec/sarthak/caduceus/')
from src.models.sequence.striped_backbone import StripedMambaBackbone
import torch

model = StripedMambaBackbone(
    d_model=512,
    n_blocks=4,
    d_in=128,
    global_pooling=128,
    transformer_pooling=4,
    mode='striped',
    ssm_per_transformer=3,
    # Hydra config
    d_state=64,
    d_conv=7,
    expand=2,
    headdim=64,
    ngroups=8,
    chunk_size=256,
    use_mem_eff_path=True,
    # Transformer config
    head_dim=64,
    use_rope=True,
    use_flash_attn=True,
    use_gating=True,
    attention_dropout=0.0,
    use_enformer_bias=False,
    pos_emb_dim=32,
    # Shared config
    expansion_factor=2,
    norm='rms',
    dropout=0.1,
    mlp_activation='gelu',
    mlp_dropout=0.0,
    fused_mlp=False,
    hydra_use_mlp=False,
    checkpoint_blocks=False,
    residual_in_fp32=True,
    rescale_prenorm_residual=True,
    zero_linear_biases=True,
    # Sampling config
    downsample_kernel_size=5,
    global_sampling_channel_scale=2,
    global_sampling_start_channels=None,
    global_sampling_grow_channels=None,
    transformer_sampling_channel_scale=1,
    transformer_sampling_start_channels=None,
    transformer_sampling_grow_channels=None,
    upsample_residual_scale_init=0.9,
    sampling_norm_type='rms',
    sampling_use_weight_std=True,
    sampling_checkpoint=True,)
model.to('cuda').bfloat16()
num_params = sum(p.numel() for p in model.parameters())
print(f'Number of parameters: {num_params:,}')
#it does technically run, but training I think use batch size 2!
import gc
import time
torch.cuda.empty_cache()
gc.collect()

x = torch.randn(2, 2**21, 128, device='cuda', dtype=torch.bfloat16)
#warmup
for _ in range(2):
    out = model(x)

t0 = time.perf_counter()
for _ in range(10):
    out = model(x)
    out.mean().backward()
torch.cuda.synchronize()
print((time.perf_counter() - t0) / 10 * 1000, "ms")

Number of parameters: 92,526,351
3826.3149439822882 ms


In [ ]:
#oh definitely do the 1 less pooling!

In [ ]:
#